# Correction as Annotation: Bootstrapping a Dependency Parser for Documentary Medieval Latin

- This notebook is the entry point to the repository. 
- It runs the analysis that the paper reports on. 
- The first run also fetches and builds the derived inputs described below. 
- Nothing in this notebook requires the trained models.


In [ ]:
from bootstrapping import config
from bootstrapping.io import load_file
from bootstrapping.report.figures import apply_style

apply_style()

print(f'Project root: {config.PROJECT_ROOT}')
print(f'Models      : {config.LATIN_MODEL_DIR}')


### Uncommitted Datasets

Some of the datasets used by the notebook are not part of the repository and need to be derived. These include the **comparison treebanks** (fetched from the Universal Dependencies repositories), the **lemma lists** (built from those treebanks), and the compiled per-sentence measurements, saved to **`corpus_statistics.json`** (a 53 MB file that can be regenerated in 30 seconds, hence not committed). 

The next cell builds whatever is missing and leaves alone whatever is already there, so the first run on a fresh clone takes a few minutes. It is the equivalent of running:

```bash
python -m bootstrapping.corpora      # fetch the 7 UD treebanks
python -m bootstrapping.lemmata      # build the lemma lists
python -m bootstrapping.statistics   # compile corpus statistics
```


In [ ]:
from pathlib import Path

from bootstrapping import corpora, lemmata, statistics

# each step is skipped if its output is already there
lemma_lists = [*lemmata.INVENTORIES.values(), config.LATIN_LEMMATA_PATH, config.OCCITAN_LEMMATA_PATH]
absent = {
    'Treebanks': corpora.missing_corpora(),
    'Lemma lists': [path for path in lemma_lists if not Path(path).exists()],
    'Corpus statistics': [] if Path(config.CORPUS_STATISTICS_PATH).exists() else ['corpus_statistics.json'],
}
for name, missing in absent.items():
    print(f'{name:20} {f"{len(missing)} to build" if missing else "present"}')

if absent['Treebanks']:
    print('\nFetching the comparison treebanks, ~98 MB...')
    corpora.ensure_corpora()

if absent['Lemma lists']:
    # dictionaries=False derives the Latin reference from the committed
    # CLTK and Lewis extracts rather than the raw sources
    print('\nBuilding the lemma lists...')
    for path, entries in lemmata.build(dictionaries=False).items():
        lemmata.write_list(path, entries)

if absent['Corpus statistics']:
    print('\nCompiling corpus statistics, this writes 53 MB...')
    statistics.write_corpus_statistics(progress=False)

print('\nEvery input the notebook needs is in place.')


## 1. The Corpus

One hundred and sixty records from the DALME collection, produced in Marseille between 1258 and 1446. The text is formulaic, heavily nominal, and mixes Latin with Occitan.

`la_marseille-ud-base-fixed.conllu` is *derived* from the database export plus three ordered clean-up steps.

In [ ]:
from bootstrapping.corpus.build import FIXED_PATH, build, serialise

sentences, reports = build()
for step, report in reports.items():
    print(f'  {step:16} ' + ', '.join(f'{k}={v:,}' for k, v in report.items()))

identical = serialise(sentences) == FIXED_PATH.read_text(encoding='utf-8')
print(f'\nRebuild is byte-identical to the committed corpus: {identical}')
assert identical, 'the derived corpus no longer matches its source'


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

excluded = {'PUNCT', 'X', 'SYM', '_'}
order = ['marseille', 'ittb', 'llct', 'perseus', 'proiel', 'udante']

stats = load_file(config.CORPUS_STATISTICS_PATH)
rows = []
for corpus in order:
    counts = stats[corpus]['upos_counts']
    denominator = sum(c for tag, c in counts.items() if tag not in excluded)
    rows.append(
        {
            group: (sum(counts.get(tag, 0) for tag in tags) / denominator * 100 if denominator else 0.0)
            for group, tags in config.POS_TAG_GROUPS.items()
        },
    )
frame = pd.DataFrame(rows, index=[config.TREEBANK_NAMES[c] for c in order]) / 100.0

apply_style(profile='print', figsize=(10, 5))
axes = frame.plot(
    kind='barh',
    stacked=True,
    figsize=(10, 5),
    colormap='tab20c',
    edgecolor='#999999',
    width=0.75,
    zorder=3,
)
axes.set_yticks(range(len(order)))
axes.set_yticklabels(frame.index, fontweight='bold')
axes.invert_yaxis()
axes.tick_params(axis='x', labelsize=6)
axes.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes.set_xlim(0, 1)
axes.set_xlabel('Percentage of total words', fontweight='bold', color='#636363')
axes.set_ylabel('')
axes.legend(title='', bbox_to_anchor=(0.5, 1.18), loc='upper center', ncol=4)

minimum_labelled = 0.04
for row_index, (_, row) in enumerate(frame.iterrows()):
    left = 0.0
    for value in row:
        if value > minimum_labelled:
            axes.text(
                left + value / 2,
                row_index,
                f'{value * 100:.1f}%',
                va='center',
                ha='center',
                fontsize=5,
                color='#333333',
                fontweight='bold',
            )
        left += value

figure = axes.get_figure()
figure.tight_layout()
plt.show()

In [ ]:
stats = load_file(config.CORPUS_STATISTICS_PATH)['marseille']
print(f'Sentences            {stats["num_sentences"]:>8,}')
print(f'Tokens               {stats["num_tokens"]:>8,}')
print(f'Words (excl. PUNCT)  {stats["num_words"]:>8,}')
print(f'Unique lemmata       {stats["num_unique_lemmata"]:>8,}')
print(
    f'Words per sentence   {stats["avg_words_per_sentence"]:>8.2f}   (median {stats["median_words_per_sentence"]:.0f})'
)


The mean of under seven words per sentence is the single most important fact about this corpus. The comparison treebanks run from 11 to 27. Short, formulaic, nominal sentences are why so little annotation goes so far (and also why the baselines perform so badly).

In [ ]:
from bootstrapping.report import tables
from bootstrapping.report.display import show

show(tables.treebank_stats())


## 2. Seed Sampling

Nine disjoint batches of 200 sentences, drawn at random within length strata (136 sentences of fewer than 7 words, 46 of 7–12, 18 of 13 or more). There is **no** stratification by date, document type, or lexical content, and **no** active learning. Batch `s1` is the exception: 204 sentences, hand-built before the procedure was regularized.

In [ ]:
quota = {'<7': 0, '7-12': 0, '>=13': 0}
rows = []
for key in config.REPORTED_SEED_KEYS:
    batch = load_file(config.SEED_TRAINING_PATHS[key])
    counts = dict.fromkeys(quota, 0)
    for sentence in batch:
        words = sum(1 for t in sentence if isinstance(t['id'], int) and t['upos'] != 'PUNCT')
        counts['<7' if words < 7 else '7-12' if words <= 12 else '>=13'] += 1  # noqa: PLR2004
    rows.append((key, len(batch), counts))

print(f'{"Batch":16}{"Sents":>7}{"<7":>7}{"7-12":>7}{">=13":>7}')
for key, n, counts in rows:
    print(f'{key:16}{n:7}{counts["<7"]:7}{counts["7-12"]:7}{counts[">=13"]:7}')


## 3. The Bootstrapping Loop

Each batch is pre-annotated with the *previous* iteration's model. `s1` uses the out-of-domain ITTB baseline, it is then corrected by hand in *brat*, then pooled with everything before it to train the next
model.

Training is not re-executed here. `bootstrapping.training` prints the exact Stanza commands.

In [ ]:
from bootstrapping import training

# prints the stock Stanza 1.2.1 commands
commands = training.training_commands(
    iteration=5,
    udbase=training.default_udbase(),
    data_root='$DATA_ROOT',
    wordvec_dir='$WORDVEC_DIR',
)
print('Training iteration 5 means running:\n')
for command in commands:
    print(f'    {command}')

print('\nOr: python scripts/train_iteration.py 5 [--execute]')


### Annotation Effort

The quantity the loop is meant to drive down is not accuracy but *work*: how much of the
pre-annotation a human has to change.

In [ ]:
show(tables.annotation_effort_efficiency())

Correction burden falls from **54.3% of tokens in s1 to 18.0% in s9**, and time per sentence from 2.57 minutes to 0.50 (a 5.1× speed-up). Total annotation cost for the reported chain is about 33 hours.

## 4. Evaluation

Nineteen configurations scored against a 200-sentence gold standard (the five baseline treebanks, each with and without the domain lexicon, and the nine project iterations, which always carry it).

Re-running the whole evaluation takes about twenty minutes and needs the 1.2.1 Stanza models:

```bash
python scripts/build_ensemble_models.py     # patch the lemma models with the lexicon
python scripts/evaluate_models.py --force   # parse and score all 19 configurations
```


In [ ]:
results = load_file(config.EVALUATION_RESULTS_PATH)
lexicon = sum(1 for entry in results['data'].values() if entry['uses_lexicon'])

print(f'Generated      : {results["generated"][:10]}')
print(f'Configurations : {len(results["data"])}')
print(f'With lexicon   : {lexicon}  (the five ensembles and all nine iterations)')


## 5. Results

### The Starting Point

Off-the-shelf Latin models do poorly on this material. LAS in the 28–62 range, MLAS in the teens and twenties—unusable for scholarship.

In [ ]:
show(tables.pre_trained_performance())


### The Learning Curve

In [ ]:
show(tables.morph_curve())

In [ ]:
show(tables.syn_curve())

In [ ]:
import matplotlib.pyplot as plt

labels = [e['name'] for e in config.REPORTED_ITERATIONS]
x = range(len(labels))

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
for metric in ('UPOS', 'XPOS', 'UFeats', 'AllTags'):
    left.plot(
        x,
        [results['data'][e['key']]['evaluation'][metric]['f1'] for e in config.REPORTED_ITERATIONS],
        marker='o',
        label=metric,
    )
for metric in ('UAS', 'LAS', 'CLAS', 'MLAS'):
    right.plot(
        x,
        [results['data'][e['key']]['evaluation'][metric]['f1'] for e in config.REPORTED_ITERATIONS],
        marker='o',
        label=metric,
    )

for axis, title in ((left, 'Morphological'), (right, 'Syntactic')):
    axis.set_xticks(list(x))
    axis.set_xticklabels(['ITTB', *[f's{i}' for i in range(1, len(labels))]], rotation=0)
    axis.set_title(title)
    axis.set_ylabel('F1')
    axis.set_ylim(0, 1)
    axis.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.show()


### The Dip is Morphological

For the first iterations the project models are **worse than the out-of-domain baseline at tagging** (`UPoS` and `XPoS` below ITTB through `s3`, `Features` and `All Tags` through `s2`), while being **better at parsing from s1 onward**. `LAS` and `MLAS` never fall below the baseline at any point.

That asymmetry is the interesting part. A model trained on a few hundred sentences of formulaic inventory Latin has seen too few adpositions and too little morphological variety to tag reliably, and ITTB's 27,000 sentences of scholastic prose genuinely know more about Latin morphology. But ITTB has never seen an inventory, and the *syntax* of these documents (short, flat, nominal, list-like) is learned almost immediately.


In [ ]:
baseline = results['data']['ittb']['evaluation']
metrics = ('UPOS', 'XPOS', 'UFeats', 'AllTags', 'LAS', 'MLAS')

print('Iterations scoring BELOW the out-of-domain ITTB baseline:\n')
for metric in metrics:
    below = [
        entry['key'].split('_s')[-1]
        for entry in config.REPORTED_ITERATIONS[1:]
        if results['data'][entry['key']]['evaluation'][metric]['f1'] < baseline[metric]['f1']
    ]
    print(f'  {metric:9} {", ".join("s" + b for b in below) if below else "never"}')

print('\nMorphology recovers by s4, syntax was never behind.')


### Two Mechanisms

The headline result is easy to state wrongly. **Lemmatisation does not improve through bootstrapping.** It is supplied almost entirely by the domain lexicon (which lifts every baseline model by 13 to 24 points) and is already near its ceiling at `s1`. Tagging and parsing are what the loop actually buys.


In [ ]:
print('What the LEXICON does—baseline models, plain vs ensemble lemmatisation:\n')
print(f'  {"model":10}{"plain":>8}{"+lexicon":>10}{"gain":>8}')
for key in ('ittb', 'llct', 'perseus', 'proiel', 'udante'):
    plain = results['data'][key]['evaluation']['Lemmas']['f1']
    ensemble = results['data'][f'{key}_ens']['evaluation']['Lemmas']['f1']
    print(f'  {key:10}{plain:8.3f}{ensemble:10.3f}{ensemble - plain:+8.3f}')

print('\n\nWhat the LOOP does—project iterations, all of which already use the lexicon:\n')
print(f'  {"iteration":12}{"Lemmas":>9}{"UPOS":>8}{"LAS":>8}{"MLAS":>8}')
for entry in config.REPORTED_ITERATIONS[1:]:
    ev = results['data'][entry['key']]['evaluation']
    print(
        f'  {entry["key"]:12}{ev["Lemmas"]["f1"]:9.3f}{ev["UPOS"]["f1"]:8.3f}{ev["LAS"]["f1"]:8.3f}{ev["MLAS"]["f1"]:8.3f}'
    )

first = results['data'][config.REPORTED_SEED_KEYS[0]]['evaluation']
last = results['data'][config.REPORTED_SEED_KEYS[-1]]['evaluation']
print('\n  s1 -> s9 change:')
for metric in ('Lemmas', 'UPOS', 'LAS', 'MLAS'):
    print(f'    {metric:8}{last[metric]["f1"] - first[metric]["f1"]:+7.3f}')


### Against the Baselines

The final reported model beats every off-the-shelf treebank model on this material, on an order of magnitude less training data, because the data is *in domain*.

In [ ]:
show(tables.cross_treebank_performance())


## 6. Where the Remaining Errors Are

Two views, one by sentence construction, and one by how long an error survives.


In [ ]:
# takes ~35s since it classifies all 200 gold sentences by construction type
show(tables.construction_specific_improvements())


## 7. Regenerating Everything

Only the last block needs the trained models.

```bash
python -m bootstrapping.corpora                 # fetch the 7 comparison treebanks
python -m bootstrapping.lemmata                 # build the lemma lists
python -m bootstrapping.statistics              # corpus_statistics.json, 53 MB
python -m bootstrapping.corpus.build            # rebuild the corrected corpus
python -m bootstrapping.report.tables           # the 10 tables  -> build/tables/
python -m bootstrapping.report.paper_figures    # the 6 figures  -> build/figures/

python scripts/fetch_models.py                  # the nine iterations from Zenodo (~900 MB)
python scripts/build_ensemble_models.py         # lexicon-patched lemma models
python scripts/evaluate_models.py --force       # all 19 configurations (~20 min)
python scripts/zero_shot_baseline.py            # current-Stanza comparison (separate venv)
```